# 01 — Data Cleaning
Load the raw Amazon reviews, inspect what's actually in them, produce clean text.

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import pandas as pd
import re, unicodedata
from bparadigm.paths import AMAZON_RAW, AMAZON_CLEAN, PROCESSED

In [3]:
df = pd.read_csv(AMAZON_RAW, engine="python", on_bad_lines="skip")
df.shape

(21214, 9)

In [4]:
df.columns.tolist()

['Reviewer Name',
 'Profile Link',
 'Country',
 'Review Count',
 'Review Date',
 'Rating',
 'Review Title',
 'Review Text',
 'Date of Experience']

In [5]:
df.head(5)

,Reviewer Name,Profile Link,Country,Review Count,Review Date,Rating,Review Title,Review Text,Date of Experience
0,Eugene ath,/users/66e8185ff1598352d6b3701a,US,1 review,2024-09-16T13:44:26.000Z,Rated 1 out of 5 stars,A Store That Doesn't Want to Sell Anything,"I registered on the website, tried to order a ...","September 16, 2024"
1,Daniel ohalloran,/users/5d75e460200c1f6a6373648c,GB,9 reviews,2024-09-16T18:26:46.000Z,Rated 1 out of 5 stars,Had multiple orders one turned up and…,Had multiple orders one turned up and driver h...,"September 16, 2024"
2,p fisher,/users/546cfcf1000064000197b88f,GB,90 reviews,2024-09-16T21:47:39.000Z,Rated 1 out of 5 stars,I informed these reprobates,I informed these reprobates that I WOULD NOT B...,"September 16, 2024"
3,Greg Dunn,/users/62c35cdbacc0ea0012ccaffa,AU,5 reviews,2024-09-17T07:15:49.000Z,Rated 1 out of 5 stars,Advertise one price then increase it on website,I have bought from Amazon before and no proble...,"September 17, 2024"
4,Sheila Hannah,/users/5ddbe429478d88251550610e,GB,8 reviews,2024-09-16T18:37:17.000Z,Rated 1 out of 5 stars,If I could give a lower rate I would,If I could give a lower rate I would! I cancel...,"September 16, 2024"


In [6]:
df["Rating"].value_counts(dropna=False)

Rating
Rated 1 out of 5 stars    13123
Rated 5 out of 5 stars     4528
Rated 4 out of 5 stars     1292
Rated 2 out of 5 stars     1227
Rated 3 out of 5 stars      885
None                        159
Name: count, dtype: int64

In [7]:
df["stars"] = df["Rating"].astype(str).str.extract(r"(\d+)\s+out").astype("float")
df["stars"].value_counts(dropna=False).sort_index()

stars
1.0    13123
2.0     1227
3.0      885
4.0     1292
5.0     4528
NaN      159
Name: count, dtype: int64

In [8]:
df["Review Text"].isna().sum()                        # missing reviews
df["Review Text"].astype(str).str.len().describe()    # how long?
df["Review Text"].duplicated().sum()                  # repeated reviews

np.int64(806)

In [9]:
_URL = re.compile(r"https?://\S+|www\.\S+")
_WS  = re.compile(r"\s+")

def clean_text(t):
    if not isinstance(t, str):
        return ""
    t = unicodedata.normalize("NFKC", t)   # normalise smart quotes / accents
    t = _URL.sub(" ", t)                    # strip urls
    return _WS.sub(" ", t).strip()          # collapse whitespace

In [10]:
df["text"] = df["Review Text"].map(clean_text)
df = df[df["text"].str.len() > 0]
df.shape

(21055, 11)

In [11]:
before = len(df)
df = df.drop_duplicates(subset="text")
print(f"dropped {before - len(df)} duplicate reviews")
df.shape

dropped 649 duplicate reviews


(20406, 11)

In [12]:
PROCESSED.mkdir(parents=True, exist_ok=True)
df[["text", "stars"]].to_csv(AMAZON_CLEAN, index=False)
print("saved:", AMAZON_CLEAN.exists(), AMAZON_CLEAN)
df[["text", "stars"]].shape

saved: True /Users/USER/code/shreya-g01/brandparadigm/bparadigm/raw_data/processed/amazon_clean.csv


(20406, 2)